# Clustering and Operating-State Identification

This notebook compares K-Means and DBSCAN to identify groups of similar turbine operating behaviour. The selected clustering result is used to identify a normal-operation group for the regression stage.

In [1]:
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

features = pd.read_csv("../out/01_preprocessing/wind_features.csv")
cluster_input = pd.read_csv("../out/01_preprocessing/cluster_input_scaled.csv")

cluster_columns = [
    "wind_speed_z",
    "power_kw_z",
    "power_ratio_z",
    "residual_kw_z",
]

X_cluster = cluster_input[cluster_columns]

print("Features:", features.shape)
print("Cluster input:", cluster_input.shape)

Features: (50530, 21)
Cluster input: (40635, 8)


## 1. K-Means Clustering

K-Means assigns every operating-region record to one of a specified number of clusters. Values of `k` from 3 to 6 are tested, and the silhouette score is used to compare cluster separation.

In [2]:
kmeans_results = []

for k in range(3, 7):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X_cluster)

    # Use a sample to calculate the score faster
    sample_size = min(10000, len(X_cluster))
    sample_index = X_cluster.sample(
        n=sample_size,
        random_state=42
    ).index

    score = silhouette_score(
        X_cluster.loc[sample_index],
        labels[sample_index]
    )

    kmeans_results.append({
        "method": "KMeans",
        "parameters": f"k={k}",
        "clusters": k,
        "noise_percent": 0,
        "silhouette_score": score
    })

kmeans_results = pd.DataFrame(kmeans_results)
kmeans_results

,method,parameters,clusters,noise_percent,silhouette_score
0,KMeans,k=3,3,0,0.513458
1,KMeans,k=4,4,0,0.525215
2,KMeans,k=5,5,0,0.467754
3,KMeans,k=6,6,0,0.410633


### Select the best K-Means configuration

The best K-Means configuration is the value of `k` with the highest silhouette score. The cluster labels are arbitrary identifiers and do not have an inherent order.

In [3]:
best_k = int(
    kmeans_results
    .sort_values("silhouette_score", ascending=False)
    .iloc[0]["clusters"]
)

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20
)

kmeans_labels = kmeans.fit_predict(X_cluster)

print("Best K-Means k:", best_k)
print(pd.Series(kmeans_labels).value_counts().sort_index())

Best K-Means k: 4
0    21701
1    15381
2     2266
3     1287
Name: count, dtype: int64


## 2. DBSCAN Clustering

DBSCAN groups dense observations and can label isolated observations as noise. The `eps` parameter controls neighbourhood size and `min_samples` controls the minimum number of nearby observations required to form a dense region.

In [4]:
dbscan_results = []

for eps in [0.3, 0.5, 0.7, 0.9, 1.1]:
    model = DBSCAN(
        eps=eps,
        min_samples=10,
        n_jobs=-1
    )

    labels = model.fit_predict(X_cluster)

    non_noise = labels != -1
    number_of_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_percent = (~non_noise).mean() * 100

    if number_of_clusters >= 2 and non_noise.sum() > number_of_clusters:
        sample_index = X_cluster.loc[non_noise].sample(
            n=min(10000, non_noise.sum()),
            random_state=42
        ).index

        score = silhouette_score(
            X_cluster.loc[sample_index],
            labels[sample_index]
        )
    else:
        score = np.nan

    dbscan_results.append({
        "method": "DBSCAN",
        "parameters": f"eps={eps}, min_samples=10",
        "clusters": number_of_clusters,
        "noise_percent": noise_percent,
        "silhouette_score": score
    })

dbscan_results = pd.DataFrame(dbscan_results)
dbscan_results

,method,parameters,clusters,noise_percent,silhouette_score
0,DBSCAN,"eps=0.3, min_samples=10",4,0.243632,0.615082
1,DBSCAN,"eps=0.5, min_samples=10",1,0.024609,NaN
2,DBSCAN,"eps=0.7, min_samples=10",1,0.000000,NaN
3,DBSCAN,"eps=0.9, min_samples=10",1,0.000000,NaN
4,DBSCAN,"eps=1.1, min_samples=10",1,0.000000,NaN


## 3. Compare K-Means and DBSCAN

The silhouette score measures how well separated the clusters are. DBSCAN is also checked for a reasonable number of clusters and an acceptable noise percentage. A high score is not useful if nearly all records are assigned to one cluster or labelled as noise.

In [5]:
clustering_results = pd.concat(
    [kmeans_results, dbscan_results],
    ignore_index=True
)

clustering_results.sort_values(
    "silhouette_score",
    ascending=False
)

,method,parameters,clusters,noise_percent,silhouette_score
4,DBSCAN,"eps=0.3, min_samples=10",4,0.243632,0.615082
1,KMeans,k=4,4,0.000000,0.525215
0,KMeans,k=3,3,0.000000,0.513458
2,KMeans,k=5,5,0.000000,0.467754
3,KMeans,k=6,6,0.000000,0.410633
5,DBSCAN,"eps=0.5, min_samples=10",1,0.024609,NaN
6,DBSCAN,"eps=0.7, min_samples=10",1,0.000000,NaN
7,DBSCAN,"eps=0.9, min_samples=10",1,0.000000,NaN
8,DBSCAN,"eps=1.1, min_samples=10",1,0.000000,NaN


## 4. Select the DBSCAN Configuration

DBSCAN with `eps=0.3` and `min_samples=10` was selected because it produced four clusters and the highest observed silhouette score of approximately `0.615`. The `-1` label represents DBSCAN noise.

In [6]:
best_dbscan = DBSCAN(
    eps=0.3,
    min_samples=10,
    n_jobs=-1
)

cluster_input["cluster"] = best_dbscan.fit_predict(X_cluster)

print(cluster_input["cluster"].value_counts().sort_index())

cluster
-1       99
 0    40476
 1       42
 2       10
 3        8
Name: count, dtype: int64


## 5. Apply Cluster Labels to the Full Dataset

The clustering input contains only records in the operating region, where wind speed is at least 3.5 m/s. Labels are joined back to all 50,530 records using `row_id`.

- `-2` is used for records below the operating range.
- `-1` represents DBSCAN noise.
- Non-negative values represent DBSCAN clusters.

In [7]:
features["cluster"] = -2

features.loc[
    cluster_input["row_id"],
    "cluster"
] = cluster_input["cluster"].to_numpy()

features["cluster_state"] = features["cluster"].astype(str)

features.loc[
    features["cluster"] == -2,
    "cluster_state"
] = "below_operating_range"

features.loc[
    features["cluster"] == -1,
    "cluster_state"
] = "dbscan_noise"

features["cluster_state"].value_counts()

cluster_state
0                        40476
below_operating_range     9895
dbscan_noise                99
1                           42
2                           10
3                            8
Name: count, dtype: int64

## Save the Clustering Outputs

The labelled full dataset is saved for the regression notebook. The next stage loads this file and trains expected-power models using normal-operation records only.

In [8]:
features.to_csv("../out/02_clustering/wind_features_clustered.csv", index = False)
cluster_input.to_csv("../out/02_clustering/cluster_input_dbscan.csv", index = False)

## 6. Interpret the Clusters

Cluster behaviour is summarised using record count, power output, power ratio, residual power, and manufacturer-baseline shortfall.

In [9]:
cluster_summary = (
    features[features["cluster"] >= 0]
    .groupby("cluster")
    .agg(
        records=("cluster", "size"),
        median_power_kw=("power_kw", "median"),
        median_power_ratio=("power_ratio", "median"),
        median_residual_kw=("residual_kw", "median"),
        mean_power_ratio=("power_ratio", "mean"),
        total_shortfall_mwh=(
            "shortfall_kwh",
            lambda values: values.sum() / 1000
        )
    )
    .sort_values("median_power_ratio", ascending=False)
)

cluster_summary

,records,median_power_kw,median_power_ratio,median_residual_kw,mean_power_ratio,total_shortfall_mwh
cluster,,,,,,
0,40476,1340.328980,0.893433,105.484484,0.836309,1533.391378
2,10,980.303223,0.272306,2619.696777,0.263647,4.418120
3,8,765.535156,0.212649,2834.464844,0.212007,3.782366
1,42,384.966507,0.106935,3208.862000,0.112522,22.319044


### Compare clusters with rule-based reference states

The rule-based states are used only as a reference to help interpret the unsupervised clusters. They are not used as training labels.

In [10]:
pd.crosstab(
    features["cluster_state"],
    features["rule_state"],
    normalize="index"
).round(3)

rule_state,below_operating_range,near_curve,underperforming,zero_output
cluster_state,,,,
0,0.0,0.771,0.175,0.055
1,0.0,0.000,1.000,0.000
2,0.0,0.000,1.000,0.000
3,0.0,0.000,1.000,0.000
below_operating_range,1.0,0.000,0.000,0.000
dbscan_noise,0.0,0.061,0.929,0.010


## 7. Select the Normal-Operation Cluster

Cluster `0` is selected as normal operation because it contains the majority of operating-region records and has the highest median power ratio. This selection is based on behaviour, not the numerical cluster label itself.

In [11]:
normal_cluster = int(
    cluster_summary["median_power_ratio"].idxmax()
)

print("Selected normal cluster:", normal_cluster)

Selected normal cluster: 0


In [12]:
features.loc[
    features["cluster"] == normal_cluster,
    "rule_state"
].value_counts(normalize=True)

rule_state
near_curve         0.770605
underperforming    0.174573
zero_output        0.054823
Name: proportion, dtype: float64

In [13]:
normal_cluster = 0

features.loc[
    features["cluster"] == normal_cluster,
    "rule_state"
].value_counts(normalize=True).round(3)

rule_state
near_curve         0.771
underperforming    0.175
zero_output        0.055
Name: proportion, dtype: float64

In [14]:
features["cluster_state"].value_counts()

cluster_state
0                        40476
below_operating_range     9895
dbscan_noise                99
1                           42
2                           10
3                            8
Name: count, dtype: int64

In [15]:
normal_data = features[
    features["cluster"] == normal_cluster
].copy()

print(normal_data.shape)

(40476, 23)


# Handover Notes

## Completed work

This notebook completed the unsupervised clustering stage.

### Input data

- Full engineered dataset: `out/01_preprocessing/wind_features.csv`
- Scaled operating-region data: `out/01_preprocessing/cluster_input_scaled.csv`
- Operating region: wind speed >= 3.5 m/s

### Clustering decision

K-Means and DBSCAN were compared using silhouette scores. DBSCAN with `eps=0.3` and `min_samples=10` was selected because it achieved a silhouette score of approximately `0.615`, compared with `0.525` for the best K-Means configuration (`k=4`).

DBSCAN produced four clusters and 24.36% noise within the operating-region data. Cluster `0` was selected as normal operation because it contained 40,476 records and had the highest median power ratio, approximately `0.893`.

The clusters are inferred operating states, not confirmed physical faults. The result is highly imbalanced, so the small clusters and DBSCAN noise should be interpreted cautiously.

## Files produced

- `out/02_clustering/wind_features_clustered.csv`
- `out/02_clustering/cluster_input_dbscan.csv`

## Next steps

The next person should:

1. Load `out/02_clustering/wind_features_clustered.csv`.
2. Use `cluster == 0` as the normal-operation training subset.
3. Compare Random Forest and XGBoost regression models.
4. Use `wind_speed`, `wd_sin`, and `wd_cos` as predictors and `power_kw` as the target.
5. Use month-blocked cross-validation.
6. Predict expected power for all records and estimate energy loss and CO2 impact.